In [145]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import pandas as pd

import sys

sys.path.append('T:\EL_experiment\Codes\CCEP_human\Python_Analysis/py_functions')

from scipy.stats import norm
from tkinter import *
import scipy
from scipy import signal

import platform
from glob import glob
from scipy.spatial import distance
import basic_func as bf
from scipy.integrate import simps
from numpy import trapz
import tqdm
from matplotlib.patches import Rectangle
from pathlib import Path
import statsmodels.api as sm
import statsmodels.formula.api as smf

#
from scipy.signal import hilbert, butter, filtfilt
import scipy.stats as stats
from tqdm.notebook import trange, tqdm
import load_summary as ls
cwd             = os.getcwd()

##all 
cond_vals   = np.arange(4)
cond_labels = ['BM', 'BL', 'Fuma', 'Benzo']
cond_colors = ['#494159','#594157', "#F1BF98","#8FB996"]
dist_groups = np.array([[0,15],[15,30],[30,5000]])
dist_labels = ['local (<15 mm)', 'short (<30mm)', 'long']
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab

sys.path.append('T:\\EL_experiment\Codes\CCEP_human\Python_Analysis\py_functions')

In [81]:
CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
tab_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='plot')
tab_region = tab_region.sort_values('Order').reset_index(drop=True)
regions_all = tab_region.Area.values
region_col = tab_region.color.values

CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
all_region = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')

In [83]:
plt.rcParams.update({
    'font.family': 'arial',
    'font.size': 12,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 9,
    'svg.fonttype': 'none',
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 9,
    'figure.titlesize': 10
})

In [84]:
sys.path.append('T:\\EL_experiment\Codes\CCEP_human\Python_Analysis\py_functions')
sys.path.append('X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram')
sys.path.append('T:\\EL_experiment\Codes\\UBELIX_EvM\\NMF\\functions')
path_connectogram = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram'
from main_script import ConnectogramPlotter
cwp = os.getcwd()

In [85]:
CIRC_AREAS_FILEPATH = "X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\\circ_areas.xlsx"
color_regions = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='plot')
atlas = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')

In [86]:
subjs = ["EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016", "EL019", "EL020", "EL021",
         "EL022", "EL024", "EL026", "EL027", "EL028"]

In [87]:
import load_summary as ls
sub_path  ='X:\\4 e-Lab\\' # y:\\eLab
df = ls.get_connections(subjs, sub_path)
#plt.savefig('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\BM\\ALL_BM\\BM_all_4.svg')

In [88]:
plotter = ConnectogramPlotter()
import matplotlib.gridspec as gridspec

In [89]:
# Custom function to adjust individual axes
def adjust_ax(ax, visible=False, rem_ticks =True):
    ax.spines['top'].set_visible(visible)
    ax.spines['right'].set_visible(visible)
    ax.spines['bottom'].set_visible(visible)
    ax.spines['left'].set_visible(visible)
    if rem_ticks:
        ax.get_xaxis().set_ticks([])
        ax.get_yaxis().set_ticks([])

In [90]:
stop

NameError: name 'stop' is not defined

In [11]:
data_mean = df.groupby(['Subj','StimR', 'ChanR','StimSR', 'ChanSR'], as_index = False)[['Sig', 'd', 'DI', 'delay']].mean()
for subj in subjs:
    mapping = {region: i for i, region in enumerate(pd.concat([data_mean.loc[data_mean.Subj ==subj, 'ChanSR'], data_mean.loc[data_mean.Subj ==subj, 'StimSR']]).unique())}
    data_mean.loc[data_mean.Subj ==subj, 'Stim'] = data_mean.loc[data_mean.Subj ==subj, 'StimSR'].map(mapping)
    data_mean.loc[data_mean.Subj ==subj, 'Chan'] = data_mean.loc[data_mean.Subj ==subj, 'ChanSR'].map(mapping)
data_mean = data_mean.rename(columns={"ChanSR": "ChanA", "StimSR": "StimA"}, errors="raise")

In [12]:
data_plot_all = data_mean[(data_mean.DI>0.3)&(data_mean.d>20)].reset_index(drop=True)

In [13]:
df_quant = df[(df.DI>0.1)&(df.Sig>0)&(df.True_peak==1)&(df.Num_sig_trial>10)&(df.d>20)].reset_index(drop=True)

In [ ]:
df_DI = df[(df.d>20)].reset_index(drop=True)

In [ ]:
area ='Hippocampus'
df_DI.loc[(np.isin(df_DI.StimR, area)), 'DI'].median()

In [ ]:
df_plot =data_plot_all.reset_index(drop=True)

## Conenctogram by Brain Region

In [ ]:
# Define the area you're focusing on
for area in regions_all:

    # Set up a gridspec with custom layout
    fig = plt.figure(figsize=(8,6))  # Adjust the figure size as needed
    gs = gridspec.GridSpec(2, 8, height_ratios=[1, 6], width_ratios=[0.4, 1, 0.5, 1, 0.3, 1, 0.5, 1])
    plt.suptitle(area)

    # Loop through 'l' and 'r' conditions
    for h_ix, h in enumerate(['l', 'r']):
        df_plot['show'] = 0
        # Logic to filter data based on 'l' or 'r' condition
        condition = (df_plot.delay < 0.02) if h == 'l' else (df_plot.delay >= 0.02)
        df_plot.loc[condition & ((np.isin(df_plot.StimR, area)) | (np.isin(df_plot.ChanR, area))), 'show'] = 1
        condition = (df_quant.delay < 0.02) if h == 'l' else (df_quant.delay >= 0.02)
        df_quant_area = df_quant[condition & ((np.isin(df_quant.StimR, area)) | (np.isin(df_quant.ChanR, area)))].reset_index(drop=True)
        label = 'direct' if h == 'l' else 'indirect'

        df_quant_area['direction'] ='outgoing'
        df_quant_area.loc[df_quant_area.ChanR==area, 'direction'] ='incoming'

        # Loop for histogram plotting
        for ix_hist, metric in enumerate(['peak_latency', 'DI']):
            ax_hist = fig.add_subplot(gs[0, 4*h_ix + ix_hist*2+1])
            sns.histplot(x=metric, data=df_quant_area, hue='direction', ax=ax_hist, multiple='stack', legend=False, hue_order=['incoming', 'outgoing'], palette=['#723783', '#69A664'])
            adjust_ax(ax_hist, visible=False, rem_ticks=False)  # Adjust axes appearance
            ax_hist.set_xlabel('Peak Latency [s]' if metric == 'peak_latency' else 'DI')
            ax_hist.set_ylabel('Count' if ix_hist == 0 and h_ix == 0 else '')

            if (ix_hist==0):
                ax_title = fig.add_subplot(gs[0, 4*h_ix+2])
                ax_title.set_title(label+' connections N='+str(len(df_quant_area)))
                adjust_ax(ax_title, visible=False)

        # Logic to sort and prepare df_plot
        df_plot['sort'] = 0
        df_plot['lw'] = 0.5
        df_plot.loc[np.isin(df_plot.ChanR, area), 'sort'] = 1
        df_plot = df_plot.sort_values(by=['sort']).reset_index(drop=True)

        if np.sum(df_plot.show) > 0:
            ax_plotter = fig.add_subplot(gs[1, h_ix*4+1:h_ix*4+4])
            os.chdir(path_connectogram)
            plotter.load_data(df_plot, ax_plotter, 'path_save', show_label=h_ix)
            plotter.show_plot(area + ' -- ' + label + ' Connections', 'area')
            if h_ix == 0:
                ax_plotter.set_xlim(ax_plotter.get_xlim()[::-1])
    ax_title = fig.add_subplot(gs[1, 4])
    ax_title.set_title('Illustration of Connections* DI > 0.3')
    adjust_ax(ax_title, False)
    # Adding the boxplot to the left of the bottom row
    ax_boxplot = fig.add_subplot(gs[1, 0])
    sns.boxplot(y='DI', data=df_DI.loc[np.isin(df_DI.StimR, area)], ax=ax_boxplot, width=0.5, fliersize=1, palette="PiYG")
    adjust_ax(ax_boxplot, True)  # Make the boxplot axes invisible
    ax_boxplot.set_yticks([-1,0,1])
    ax_boxplot.set_ylabel('Directionaliy Index across all long-range connections')
    # Make the boxplot transparent
    #for patch in ax_boxplot.artists:
    #    patch.set_facecolor('PiYG')

    # Add background color scale to the boxplot axes
    #ylim = ax_boxplot.get_ylim()  # Get current y limits
    #img = np.array([np.linspace(-1, 1, 100)])  # For color scale
    #ax_boxplot.imshow(img, aspect='auto', extent=(0, 1, ylim[0], ylim[1]), cmap="PiYG", alpha=0.3, origin='lower')

    fig.subplots_adjust(wspace=0.001, hspace=0.5)

    path_save ='X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\BM\\ALL_BM\\Connectogram_'+area+'.svg'
    # plt.savefig(path_save)
    plt.show()

## Connecttogram by Tract

In [14]:
atlas_all = ['AF_L', 'AF_R', 'C_FPH_L', 'C_FPH_R', 'C_FP_L', 'C_FP_R',
       'C_PHP_L', 'C_PHP_R', 'C_PH_L', 'C_PH_R', 'C_PO_L', 'C_PO_R','EMC_L', 'EMC_R', 'FAT_L', 'FAT_R', 'IFOF_L',
       'IFOF_R', 'ILF_L', 'ILF_R','MdLF_L', 'MdLF_R', 'PAT_L',
       'PAT_R', 'SLF1_L', 'SLF1_R', 'SLF2_L', 'SLF2_R', 'SLF3_L',
       'SLF3_R', 'VOF_L', 'VOF_R']

# Remove the _L and _R suffixes and get unique labels
atlas_all = set(label[:-2] for label in atlas_all)

In [15]:
def adding_destrieux(data_con, pair=1, area='Destrieux'):
    # area == 'Region' or 'Area'
    data_con = data_con.reset_index(drop=True)
    CIRC_AREAS_FILEPATH = 'X:\\4 e-Lab\e-Lab shared code\Softwares\Connectogram\circ_areas.xlsx'
    atlas = pd.read_excel(CIRC_AREAS_FILEPATH, sheet_name='atlas')
    if pair:
        for subregion in np.unique(data_con[['StimA', 'ChanA']]):
            region = atlas.loc[atlas.Abbreviation == subregion, area].values
            
            if len(region) > 0:
                if region[0] == "entorhinal_cortex":
                    region[0] = 'Entorhinal' 
                elif region[0] == "AnteroHippocampus":
                     region[0] = 'Hippocampus'
                elif region[0] == "PosteroHippocampus":
                     region[0] = 'Hippocampus'
                data_con.loc[data_con.StimA == subregion, 'StimR'] = region[0]
                data_con.loc[data_con.ChanA == subregion, 'ChanR'] = region[0]
            else:
                
                print(subregion)
                data_con.loc[data_con.StimA == subregion, 'StimR'] = 'U'
                data_con.loc[data_con.ChanA == subregion, 'ChanR'] = 'U'
    else:
        for subregion in np.unique(data_con[['ChanA']]):
            region = atlas.loc[atlas.Abbreviation == subregion, area].values
            if len(region) > 0:
                data_con.loc[data_con.ChanA == subregion, 'ChanR'] = region[0]
            else:
                data_con.loc[data_con.ChanA == subregion, 'ChanR'] = 'U'
    return data_con

In [16]:
def adding_distance_tracts(df, df_atlas, atlas = False):
    # Correct renaming of columns with a dictionary and using inplace or assignment
    df_atlas = df_atlas.rename(columns={'Count': 'tract_num', 'Length': 'tract_dist'})
    for col in ['tract_num', 'tract_dist']:
        if col in df:
            df = df.drop(columns=col)
    # Ensure that 'Stim' and 'Chan' are columns to be merged on both DataFrames
    # Correct the usage of selecting multiple columns from df_atlas
    if atlas:
        df = df.merge(df_atlas, on=['Stim', 'Chan'],
                      how='left').reset_index(drop=True)
        
    else:
        df = df.merge(df_atlas[['Stim', 'Chan', 'tract_num', 'tract_dist']], on=['Stim', 'Chan'],
                      how='left').reset_index(drop=True)
    return df

In [159]:
subjs = ["EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016", "EL019", "EL020", "EL021", "EL022", "EL024",
         "EL026", "EL027", "EL028"]


sub_path  ='X:\\4 e-Lab\\'
path_save = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', 'Across', 'WM_tracts', 'Figures')
con_all = ls.get_connections(subjs, sub_path)
con_all.loc[con_all.True_peak<0.5, 'Sig'] = 0
# Main loop
start_r = 20
end_r = 20
print(f"Processing for start_r={start_r} and end_r={end_r}")
# Load and process data for each subject
data_all = pd.DataFrame()
for subj in subjs:
    print(subj, end='\r')
    # WM_dist_file = os.path.join(sub_path, 'Patients', subj, 'Electrodes','Tracts',
    #                             f"{subj}_tracts_len_contacts_s{start_r}_e{end_r}.csv")
    # WM_count_file = os.path.join(sub_path, 'Patients', subj, 'Electrodes','Tracts',
    #                              f"{subj}_tracts_num_contacts_s{start_r}_e{end_r}.csv")
    file = os.path.join(sub_path, 'Patients', subj, 'Electrodes', 'Tracts',
                        f"{subj}_tracts_contacts_s{start_r}_e{end_r}_Region.csv")
    # self.
    if os.path.isfile(file):
        # tracts_distances = pd.read_csv(WM_dist_file).values[:, 1:]
        # tracts_counts = pd.read_csv(WM_count_file).values[:, 1:]
        tract_matrix = pd.read_csv(file)
        data_subj = adding_distance_tracts(con_all[con_all.Subj == subj].reset_index(drop=True), tract_matrix, True)
        data_all = pd.concat([data_all, data_subj], ignore_index=True)


Processing for start_r=20 and end_r=20


In [160]:
data_all = group_connections(data_all)

In [161]:
data_all.loc[data_all.True_peak<0.5, 'Sig'] = 0

In [149]:
col_keep = ['Subj', 'Stim', 'Chan', 'LL', 'Sig', 'DI', 'delay', 'peak_latency', 'Hemi','Group','AF_L', 'AF_R', 'C_FPH_L', 'C_FPH_R', 'C_FP_L', 'C_FP_R',
       'C_PHP_L', 'C_PHP_R', 'C_PH_L', 'C_PH_R', 'C_PO_L', 'C_PO_R','EMC_L', 'EMC_R', 'FAT_L', 'FAT_R', 'IFOF_L',
       'IFOF_R', 'ILF_L', 'ILF_R','MdLF_L', 'MdLF_R', 'PAT_L',
       'PAT_R', 'SLF1_L', 'SLF1_R', 'SLF2_L', 'SLF2_R', 'SLF3_L',
       'SLF3_R', 'VOF_L', 'VOF_R','tract_dist', 'tract_num']
file_save = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', 'Across', 'Connecto', 'Data', 'con_across.csv')
df_save = data_all[col_keep].reset_index(drop=True)
df_save.to_csv(file_save, header =True, index =False)

In [112]:
def adding_mni_coord(lbls, subj):
    # Load Lookup from camille.
    path_gen = os.path.join(sub_path, 'Patients', subj, 'Electrodes', "Lookup.xlsx")
    lookup = pd.read_excel(path_gen, header=0, sheet_name='bipoles')

    if "type" in lookup:
        lookup = lookup[lookup['type'] == 'lead'].reset_index(drop=True)

    # Rounding coordinates
    for coord in ['native_x', 'native_y', 'native_z', 'mni_x', 'mni_y', 'mni_z']:
        lookup[coord] = np.round(lookup[coord], 2)

    for coord in ['x', 'y', 'z']:
        lbls[coord] = np.round(lbls[coord], 2)

    # Preparing for merge: Renaming lbls columns for matching with lookup
    lbls.rename(columns={'x': 'native_x', 'y': 'native_y', 'z': 'native_z'}, inplace=True)

    # Merging dataframes on rounded coordinates
    lbls = lbls.merge(lookup[['native_x', 'native_y', 'native_z', 'mni_x', 'mni_y', 'mni_z']],
                      on=['native_x', 'native_y', 'native_z'], how='left')

    return lbls

In [158]:
subjs = ["EL010", "EL011", "EL012", "EL013", "EL014", "EL015", "EL016", "EL019", "EL020", "EL021", "EL022", "EL024",
         "EL026", "EL027", "EL028"]

col_keep = ['label', 'native_x', 'native_y', 'native_z', 'mni_x', 'mni_y', 'mni_z']
sub_path  ='X:\\4 e-Lab\\'
path_save = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', 'Across', 'WM_tracts', 'Figures')
data_all = pd.DataFrame()
for subj in subjs:
    print(subj, end='\r')
    file = os.path.join(sub_path, 'Patients', subj, 'Electrodes',
                        f"{subj}_labels.xlsx")
    # self.
    if os.path.isfile(file):
        # tracts_distances = pd.read_csv(WM_dist_file).values[:, 1:]
        # tracts_counts = pd.read_csv(WM_count_file).values[:, 1:]
        lbls = pd.read_excel(file, header=0, sheet_name='BP')
        
        if "type" in lbls:
            lbls = lbls[lbls['type'] == 'SEEG'].reset_index(drop=True)
        lbls = adding_mni_coord(lbls, subj)
       
        lbls = lbls[col_keep]
        lbls.insert(0, 'c', np.arange(len(lbls)))
        lbls.insert(0, 'Subj', subj)
        #data_subj = adding_distance_tracts(con_all[con_all.Subj == subj].reset_index(drop=True), tract_matrix, True)
        data_all = pd.concat([data_all, lbls], ignore_index=True)


In [157]:
file_save = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', 'Across', 'Connecto', 'Data', 'labels_across.csv')
data_all.to_csv(file_save, header =True, index =False)

In [162]:
df_plot = data_all[(data_all.Sig>0)&(data_all.d>20)&(data_all.H==0)].reset_index(drop=True)
df_quant = data_all[(data_all.d>20)&(data_all.H==0)].reset_index(drop=True)

In [163]:
df_quant['direction'] = 'bi-directional'
df_quant.loc[df_quant.DI<=0.1, 'direction'] = 'incoming'
df_quant.loc[df_quant.DI>=0.1, 'direction'] = 'outgoing'
df_quant['abs_DI']= abs(df_quant.DI)

In [164]:
df_plot = adding_destrieux(df_plot, area = 'Region')

Unknown


In [99]:
for atlas_sel in atlas_all:
    print(atlas_sel)
    fig = plt.figure(figsize=(8,4))  # Adjust the figure size as needed
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 3])

    h_ix = 1
    h ='l'
    df_plot['show'] = 0
    df_plot.loc[(abs(df_plot[atlas_sel+'_R'])==1)|(abs(df_plot[atlas_sel+'_L'])==1), 'show'] = 1
    if np.mean(df_plot['show'])>0:
        df_quant['tract_dir'] = 'none'
        df_quant.loc[(df_quant[atlas_sel+'_R']==1)|(df_quant[atlas_sel+'_L']==1), 'tract_dir'] = 'outgoing'
        df_quant.loc[(df_quant[atlas_sel+'_R']==-1)|(df_quant[atlas_sel+'_L']==-1), 'tract_dir'] = 'incoming'
        plt.suptitle(atlas_sel+' (N='+str(np.sum(df_plot.show))+')')

        # TODO: plot CCEP onset, absolute DI , WM lenght, WM count
        for i, metric in enumerate(['abs_DI', 'delay']):
            if len(df_quant[(df_quant.tract_dir=='outgoing')])>0:
                ax_hist = fig.add_subplot(gs[0, i])
                sns.histplot(x=metric, data=df_quant[(df_quant.tract_dir=='outgoing')], ax=ax_hist, hue ='direction', multiple='stack', hue_order=['incoming', 'outgoing'], palette=['#723783', '#69A664'])
                adjust_ax(ax_hist, visible=False, rem_ticks=False)  # Adjust axes appearance
                ax_hist.set_xlabel('Onset [s]' if metric == 'delay' else 'DI')
                ax_hist.set_ylabel('Count' if i == 0 and h_ix == 0 else '')
                if i==1:
                    ax_hist.get_legend().remove()
                else:
                    ax_hist.set_title('DI = '+str(np.round(np.mean(df_quant.loc[df_quant.tract_dir=='outgoing', 'DI']),2)))

        for i, metric in enumerate(['tract_dist', 'tract_num']):
            ax_hist = fig.add_subplot(gs[1, i])
            sns.histplot(x=metric, data=df_plot[df_plot.show==1], ax=ax_hist)
            adjust_ax(ax_hist, visible=False, rem_ticks=False)  # Adjust axes appearance
            ax_hist.set_xlabel('WM tract length' if metric == 'tract_dist' else 'WM tract count')
            ax_hist.set_ylabel('Count' if i == 0 and h_ix == 0 else '')
            if metric == 'tract_num':
                ax_hist.set_xscale('log')

        df_plot['show'] = 0
        df_plot.loc[(df_plot['DI']>0.1)&((abs(df_plot[atlas_sel+'_R'])==1)|(abs(df_plot[atlas_sel+'_L'])==1)), 'show'] = 1
        # Logic to sort and prepare df_plot
        df_plot['sort'] = 0
        df_plot['lw'] = 0.5

        if np.sum(df_plot.show) > 0:
            ax_plotter = fig.add_subplot(gs[:, 2])
            os.chdir(path_connectogram)
            plotter.load_data(df_plot, ax_plotter, 'path_save', show_label=h_ix)
            plotter.show_plot(atlas_sel + ' -- DI>0.1 Connections', 'area')
            if h_ix == 0:
                ax_plotter.set_xlim(ax_plotter.get_xlim()[::-1])
        plt.tight_layout()
        path_save ='X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\BM\\ALL_BM\\Connectogram_'+atlas_sel+'.svg'
        plt.savefig(path_save)
        plt.close()

C_PO
IFOF
EMC
MdLF
SLF1
AF
C_FPH
SLF2
SLF3
FAT
C_FP
PAT
VOF
ILF
C_PHP
C_PH


<Figure size 576x288 with 0 Axes>

<Figure size 576x288 with 0 Axes>

<Figure size 576x288 with 0 Axes>

In [49]:
for atlas_sel in atlas_all:
    print(atlas_sel)
    fig = plt.figure(figsize=(8,4))  # Adjust the figure size as needed
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 3])
    
    h_ix = 1
    h ='l'
    df_plot['show'] = 0
    df_plot.loc[(abs(df_plot[atlas_sel+'_R'])==1)|(abs(df_plot[atlas_sel+'_L'])==1), 'show'] = 1
    plt.suptitle(atlas_sel+' (N='+str(np.sum(df_plot.show))+')')

    # TODO: plot CCEP onset, absolute DI , WM lenght, WM count
    for i, metric in enumerate(['DI', 'delay']):
        ax_hist = fig.add_subplot(gs[0, i])
        sns.histplot(x=metric, data=df_plot[df_plot.show==1], ax=ax_hist)
        adjust_ax(ax_hist, visible=False, rem_ticks=False)  # Adjust axes appearance
        ax_hist.set_xlabel('Onset [s]' if metric == 'delay' else 'DI')
        ax_hist.set_ylabel('Count' if i == 0 and h_ix == 0 else '')

    for i, metric in enumerate(['tract_dist', 'tract_num']):
        ax_hist = fig.add_subplot(gs[1, i])
        sns.histplot(x=metric, data=df_plot[df_plot.show==1], ax=ax_hist)
        adjust_ax(ax_hist, visible=False, rem_ticks=False)  # Adjust axes appearance
        ax_hist.set_xlabel('WM tract length' if metric == 'tract_dist' else 'WM tract count')
        ax_hist.set_ylabel('Count' if i == 0 and h_ix == 0 else '')
        if metric == 'tract_num':
            ax_hist.set_xscale('log')

    df_plot['show'] = 0
    df_plot.loc[(df_plot['DI']>0.3)&((abs(df_plot[atlas_sel+'_R'])==1)|(abs(df_plot[atlas_sel+'_L'])==1)), 'show'] = 1
    # Logic to sort and prepare df_plot
    df_plot['sort'] = 0
    df_plot['lw'] = 0.5

    if np.sum(df_plot.show) > 0:
        ax_plotter = fig.add_subplot(gs[:, 2])
        os.chdir(path_connectogram)
        plotter.load_data(df_plot, ax_plotter, 'path_save', show_label=h_ix)
        plotter.show_plot(atlas_sel + ' -- DI>0.1 Connections', 'area')
        if h_ix == 0:
            ax_plotter.set_xlim(ax_plotter.get_xlim()[::-1])
    plt.tight_layout()
    path_save ='X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\BM\\ALL_BM\\Connectogram_'+atlas_sel+'.svg'
    plt.savefig(path_save)
    plt.close()

C_PO
IFOF
EMC
MdLF
SLF1
AF
C_FPH
SLF2
SLF3
FAT
C_FP
PAT
VOF
ILF
C_PHP
C_PH


In [ ]:
## Connectogram for one subject

In [ ]:
subj ='EL011'

In [ ]:
df_plot = df[(df.Subj==subj)].reset_index(drop=True)

In [ ]:

path_patient_analysis = os.path.join(sub_path, 'EvM', 'Projects', 'EL_experiment', 'Analysis', 'Patients', subj)
path_infos = os.path.join(sub_path,'Patients', subj,'Electrodes')
lbls = pd.read_excel(os.path.join(path_infos, subj + "_labels.xlsx"), header=0, sheet_name='BP')
if "type" in lbls.columns:
    lbls = lbls[lbls.type=='SEEG']
    lbls = lbls.reset_index(drop=True)

stimlist_file = path_patient_analysis + '\\BrainMapping\\CR\\data\\stimlist_CR.csv'
stimlist = pd.read_csv(stimlist_file)
labels_all, labels_region, labels_clinic, coord_all, StimChans, StimChanSM, StimChansC, StimChanIx, stimlist = bf.get_Stim_chans(
stimlist,
lbls)

# Assuming labels_all is a list of labels in the order they should appear in the matrix
n = len(labels_all)  # Number of labels/nodes
indices = pd.MultiIndex.from_frame(df_plot[['Stim', 'Chan']])

      #for ix_metric, metric, cmap, yl in zip([0,1,2],['LL_sig', 'Sig', 'DI'],[ 'bone', cmap_metric,'PiYG'],[1,0,0]):
        # Initialize matrix with NaNs or zeros

metric = 'Sig'
M = np.full((n, n), np.nan)

# Fill the matrix with metric values
for ((stim, chan), value) in zip(indices, df_plot[metric]):
    M[stim, chan] = value



In [ ]:
# Plot
axmatrix = axs[ix_s, (ix_metric*2)]
axmatrix_col = axs[ix_s, (ix_metric*2)+1]
hemisphere = lbls['Hemisphere'].values  # Assuming this is how you determine hemisphere for each label
plot_BM(M, labels_all, hemisphere, axmatrix, cmap=cmap, plot_y = yl, ax_col = axmatrix_col)
axmatrix.set_aspect('equal')

In [ ]:
df_plot.loc[(df_plot.True_peak==0)|(df_plot.Num_sig_trial<=10), 'Sig'] = 0

In [ ]:
# Set up a gridspec with custom layout
fig = plt.figure(figsize=(6,10))  # Adjust the figure size as needed
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 0.1])
plt.suptitle('Brain Connectivity in EL011')
df_plot['show'] = 0

axmatrix = fig.add_subplot(gs[0,0])
axmatrix_col = fig.add_subplot(gs[0,1])
hemisphere = lbls['Hemisphere'].values  # Assuming this is how you determine hemisphere for each label
plot_BM(M, labels_all, hemisphere, axmatrix, cmap=cmap_metric, plot_y = 1, ax_col = axmatrix_col)
axmatrix.set_aspect('equal')


df_plot.loc[(df_plot.Sig>0.2), 'show'] = 1

# Logic to sort and prepare df_plot
df_plot['lw'] = df_plot['Sig']/10+0.3
if np.sum(df_plot.show) > 0:
    ax_plotter = fig.add_subplot(gs[1,0])
    os.chdir(path_connectogram)
    plotter.load_data(df_plot, ax_plotter, 'path_save', show_label=h_ix)
    plotter.show_plot(area + ' -- ' + label + ' Connections', 'area')
    ax_plotter.set_aspect(0.7)
    
df_plot['show'] = 0

df_plot.loc[(df_plot.Sig>0.1)&(df_plot.StimR=='Dorsofrontal'), 'show'] = 1

# Logic to sort and prepare df_plot
df_plot['lw'] = df_plot['Sig']/10+0.3
if np.sum(df_plot.show) > 0:
    ax_plotter = fig.add_subplot(gs[2,0])
    os.chdir(path_connectogram)
    plotter.load_data(df_plot, ax_plotter, 'path_save', show_label=h_ix)
    plotter.show_plot(area + ' -- ' + label + ' Connections', 'area')
    ax_plotter.set_aspect(0.7)
    
plt.savefig('X:\\4 e-Lab\EvM\Projects\EL_experiment\Analysis\Supp_figures\BM\\ALL_BM\\Map_Connectogram_'+subj+'.svg')

In [ ]:
gs = gridspec.GridSpec(2, 8, height_ratios=[1, 6], width_ratios=[0.4, 1, 0.5, 1, 0.3, 1, 0.5, 1])
    

In [ ]:
import matplotlib
# Define color values and corresponding colors
cvals  = [0, 0.1, 0.4, 0.85, 0.98]
colors = ["black","black", "firebrick", "orange", "yellow"]

# Normalize the color values to be between 0 and 1
norm = matplotlib.colors.Normalize(min(cvals), max(cvals))
norm_cvals = [norm(val) for val in cvals]

# Create tuples of normalized values and colors
tuples = list(zip(norm_cvals, colors))

# Create the colormap
cmap_metric = matplotlib.colors.LinearSegmentedColormap.from_list("", tuples)

In [ ]:
def clean_label(labels):
    """Remove spaces and numbers from labels."""
    labels = [label.replace(" ", "") for label in labels]
    labels = [re.sub(r'\d', '', label) for label in labels]
    return labels


def remove_bad_areas(M, labels, regions, hemisphere, axis=-1):
    """Filter out bad regions from the matrix and labels."""
    BAD_REGIONS = {'WM', 'OUT', 'out', 'Putamen', 'Pallidum', 'Necrosis', 'LV', 'U', 'Unknown'}

    valid_indices = [i for i, region in enumerate(regions) if region not in BAD_REGIONS]
    if axis == -1:
        M = M[valid_indices][:, valid_indices]
    elif axis == 0:
        M = M[valid_indices, :]
    else:
        M = M[:, valid_indices]
    labels = [labels[i] for i in valid_indices]
    regions = [regions[i] for i in valid_indices]
    hemisphere = [hemisphere[i] for i in valid_indices]
    return M, labels, regions, hemisphere


def get_region(labels, atlas):
    """Retrieve the region for each label from the atlas."""
    return [atlas.loc[atlas.Abbreviation == l, 'Region'].values[0] if l in atlas.Abbreviation.values else 'Unknown' for
            l in labels]


def sort_M(M, labels, regions, hemisphere, axis=-1):
    areas_sel_sort = [h + '_' + r for h, r in zip(hemisphere, regions)]
    ind = np.argsort(areas_sel_sort)

    if axis == -1:
        M = M[ind][:, ind]
    elif axis == 0:
        M = M[ind]
    else:
        M = M[:, ind]
    labels = [labels[i] for i in ind]
    regions = [regions[i] for i in ind]
    hemisphere = [hemisphere[i] for i in ind]

    return M, labels, regions
def plot_BM(M, labels, hemisphere, axmatrix, axcolor=None, cmap='hot', vlim=None, sort=1, plot_y = 1, ax_col = None):
    """Plot the connectivity map."""
    labels = clean_label(labels)
    areas = get_region(labels, atlas)
    M, labels, areas, hemisphere = remove_bad_areas(M, labels, areas, hemisphere)

    if vlim is None:
        vlim = [np.nanpercentile(M, 5), np.nanpercentile(M, 95)]
    if cmap == 'PiYG':
        cmap = plt.get_cmap('PiYG')  # Get the 'PiYG' colormap
        cmap.set_bad(color='black')  # Set NaN values to black
        vlim = [-0.95, 0.95]
        # set nan to black
    elif cmap == 'bone':
        cmap = plt.get_cmap('bone')  # Get the 'PiYG' colormap
        cmap.set_bad(color='black')  # Set NaN values to black
        vlim = [np.nanpercentile(M, 5), np.nanpercentile(M, 90)]
    else:
        cmap.set_bad(color='black')
        vlim = [0, 1]
        # set nan to black
        
    if sort:
        M, labels, areas = sort_M(M, labels, areas, hemisphere)
    labels = [h + '_' + r for h, r in zip(hemisphere, labels)]
    n_nodes = M.shape[0]

    im = axmatrix.pcolormesh(M, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    # im = axmatrix.matshow(M, aspect='auto', origin='lower', cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    axmatrix.set_xlim([-1.5, n_nodes - 0.5])
    axmatrix.set_ylim([-0.5, n_nodes + 0.5])
    
    if plot_y:
        axmatrix.set_yticks(np.arange(n_nodes) + 0.5)  # Adjust to center of the grid cell
        axmatrix.set_yticklabels(labels, fontsize=8)
        axmatrix.set_xticks(np.arange(n_nodes) + 0.5)  # Adjust to center of the grid cell
        axmatrix.set_xticklabels(labels, rotation=90, fontsize=8)
    else:
         # Remove x tick labels
        axmatrix.set_yticklabels([])  # Remove y tick labels
    # axmatrix.set_xticklabels([]) 
    # Add colored rectangles for regions
    for i in range(n_nodes):
        r = areas[i]
        color = color_regions.loc[color_regions.Area == r, 'color'].values[0]
        axmatrix.add_patch(Rectangle((i - 0.5, n_nodes - 0.5), 1, 1, alpha=1, facecolor=color))
        axmatrix.add_patch(Rectangle((-1.5, i - 0.5), 1, 1, alpha=1, facecolor=color))

    if ax_col:
        fig.colorbar(im, cax=ax_col)
        ax_col.yaxis.set_tick_params(pad=-22)